# FailureTrace — real autoresearch trials on a free GPU

Generates **real** `run.log` artifacts from `karpathy/autoresearch`, ingests them into
FailureTrace, walks the promotion ladder, and prints the resulting causal-support levels.

This is the **Level-1 plumbing validation**: it proves the parser survives a real log, a
real CUDA OOM classifies as `resource_pressure`, and the ladder reaches C3 on real
evidence. It is *not* a test of whether FailureTrace saves compute — the trial design
deliberately engineers OOMs to exercise the one deterministic, plannable route.

## Before you run

| Platform | Setup |
|---|---|
| **Kaggle** | Session options → Accelerator **GPU T4 x2** (*not* P100), Internet **On**. ~30 GPU-hr/week free. |
| **Colab** | Runtime → Change runtime type → **T4** (free) or **L4/A100** (Pro). |
| **Lightning** | Prepare data on a **free CPU Studio**, then switch the same Studio to A100. |

Runtime: ~20–40 min data prep (CPU) + ~30–45 min training (GPU).
Disk: ~10 GB. RAM: 16 GB comfortable, 8 GB workable with fewer shards.

## 1 · Environment probe → pick the profile

In [ ]:
import os, sys, shutil, subprocess, json
from pathlib import Path

def _exists(p): return Path(p).exists()

if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or _exists("/kaggle/working"):
    PLATFORM, ROOT = "kaggle", Path("/kaggle/working")
elif "google.colab" in sys.modules:
    PLATFORM, ROOT = "colab", Path("/content")
elif _exists("/teamspace/studios/this_studio"):
    PLATFORM, ROOT = "lightning", Path("/teamspace/studios/this_studio")
else:
    PLATFORM, ROOT = "local", Path.cwd()

print(f"platform : {PLATFORM}   workdir={ROOT}")
print(f"python   : {sys.version.split()[0]}")

import torch
print(f"torch    : {torch.__version__}  (cuda {torch.version.cuda})")
assert torch.cuda.is_available(), (
    "No CUDA GPU visible. autoresearch/train.py hard-codes device='cuda' "
    "(train.py:461) — it cannot run on CPU or Apple MPS."
)

CAP     = torch.cuda.get_device_capability()
GPU     = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"gpu      : {GPU}  sm_{CAP[0]}{CAP[1]}  {VRAM_GB:.1f} GiB")

try:
    RAM_GB = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
except (ValueError, OSError):
    RAM_GB = float("nan")
FREE_GB = shutil.disk_usage(ROOT).free / 1024**3
print(f"host     : {RAM_GB:.0f} GiB RAM, {FREE_GB:.0f} GiB free disk on {ROOT}")

# --- profile selection -------------------------------------------------------
#   stock : Hopper. FlashAttention-3 runs natively, no patch, bf16, 44 GB peak.
#   a100  : Ampere/Ada. patch_train.py swaps FA3 -> torch SDPA; bf16 retained.
#   t4    : Turing. + patch_t4.py: depth 8, seq 1024, batch 8, fp32 (Turing has no bf16, and fp16 overflows this trainer).
if CAP == (9, 0):
    PROFILE = "stock"
elif CAP >= (8, 0):
    PROFILE = "a100"
elif CAP == (7, 5):
    PROFILE = "t4"
else:
    raise SystemExit(
        f"Unsupported GPU sm_{CAP[0]}{CAP[1]} ({GPU}). Needs Turing (T4) or newer.\n"
        "On Kaggle: pick 'GPU T4 x2', NOT 'GPU P100' (P100 is sm_60 — no usable fp16 "
        "tensor-core path for this trainer)."
    )

print(f"\n>>> profile: {PROFILE}")
if FREE_GB < 10:
    print("!! WARNING: <10 GiB free disk. Reduce SHARDS below or free space.")

## 2 · Sizing

`prepare.py` trains the BPE tokenizer over a 1 GB character budget (`prepare.py:125`,
`text_iterator(max_chars=1_000_000_000)`) — that is the RAM-hungry step. Each shard is
**88 MiB** (92,291,918 bytes), so shard count drives both download and disk.

The reference stock run reports `peak_vram_mb: 45060.2` ≈ **44 GB** at
`DEVICE_BATCH_SIZE=128` × `MAX_SEQ_LEN=2048`. On anything smaller than an 80 GB card the
*baseline* itself would OOM — and `run_trials.py` aborts before writing failure evidence
when the baseline produces no `val_bpb`. So the batch is scaled to the card below, and
the failure/fix configs are scaled with it to keep the experimental design intact.

In [ ]:
# Shards: enough to feed the tokenizer's 1 GB char budget without exhausting RAM.
SHARDS = 4 if RAM_GB < 14 else 8
print(f"shards       : {SHARDS}  (~{SHARDS * 88 / 1024:.2f} GiB download, +1 pinned val shard)")

CUSTOM_CONFIGS = None   # None -> use run_trials.py's built-in profile configs

if PROFILE in ("stock", "a100"):
    # Reference: batch 128 -> ~44 GB. Activation memory scales ~linearly with batch;
    # weights + Muon/AdamW state are a small fixed overhead for this 50 M-param model.
    usable = VRAM_GB * 0.80 - 4.0            # headroom for fragmentation + eval
    raw = max(4.0, usable / 44.0 * 128)
    base = 1
    while base * 2 <= raw:                    # round down to a power of two
        base *= 2
    base = max(4, min(128, base))
    if base < 128:
        print(f"baseline batch: {base} (scaled down from 128 for {VRAM_GB:.0f} GiB)")
        CUSTOM_CONFIGS = [
            {"label": "baseline", "role": "baseline", "seed": 42,
             "components": ["model"], "overrides": {"DEVICE_BATCH_SIZE": base}},
            {"label": "oom-a", "role": "failure", "seed": 43,
             "components": ["model"], "overrides": {"DEVICE_BATCH_SIZE": base * 8}},
            {"label": "oom-b", "role": "failure", "seed": 44,
             "components": ["model"], "overrides": {"DEVICE_BATCH_SIZE": base * 8}},
            {"label": "fix", "role": "counterfactual", "seed": 45,
             "components": ["model"], "overrides": {"DEVICE_BATCH_SIZE": max(2, base // 4)}},
        ]
    else:
        print("baseline batch: 128 (stock)")
else:
    print("baseline batch: 4 handicapped / fix 8 (T4 profile: depth 8, seq 1024, fp32)")

print(f"\nBudget ~7 min/run x 4 runs (5 min train + torch.compile + eval) "
      f"= ~30 min GPU, plus prep.")

## 3 · Fetch the repos

In [ ]:
AUTORESEARCH = ROOT / "autoresearch"
FAILURETRACE = ROOT / "FailureTrace"
AR_PIN = "228791f"   # the commit tools/lightning/*.py patches are written against

def sh(*cmd, cwd=None, check=True):
    print("$", " ".join(str(c) for c in cmd))
    return subprocess.run([str(c) for c in cmd], cwd=cwd, check=check)

if not AUTORESEARCH.exists():
    sh("git", "clone", "--quiet", "https://github.com/karpathy/autoresearch", AUTORESEARCH)
if not (AUTORESEARCH / ".git" / "FT_PINNED").exists():
    r = subprocess.run(["git", "checkout", "--quiet", AR_PIN], cwd=AUTORESEARCH)
    if r.returncode == 0:
        (AUTORESEARCH / ".git" / "FT_PINNED").touch()
    else:
        print(f"!! could not check out {AR_PIN}; the patches may not apply cleanly.")

if not FAILURETRACE.exists():
    sh("git", "clone", "--quiet", "https://github.com/taysi-ma/FailureTrace", FAILURETRACE)

# Dependencies. torch is preinstalled on Kaggle/Colab — never reinstall it, the cu128
# wheel is ~3 GB and swapping it can break the CUDA runtime.
missing = []
for mod, pkg in (("pyarrow", "pyarrow"), ("rustbpe", "rustbpe"),
                 ("tiktoken", "tiktoken"), ("requests", "requests")):
    try:
        __import__(mod)
    except ImportError:
        missing.append(pkg)
# `kernels` is only needed on the unpatched Hopper path — patch_train.py removes the
# `from kernels import get_kernel` block entirely (train.py:20-24).
if PROFILE == "stock":
    try:
        __import__("kernels")
    except ImportError:
        missing.append("kernels>=0.11.7")

if missing:
    sh(sys.executable, "-m", "pip", "install", "-q", *missing)
sh(sys.executable, "-m", "pip", "install", "-q", "-e", str(FAILURETRACE))

# pip -e drops a .pth that site.py only processes at interpreter startup, so the running
# kernel never sees it. Put the source dir on sys.path directly, and in PYTHONPATH so the
# run_trials.py subprocess inherits it.
sys.path.insert(0, str(FAILURETRACE))
os.environ["PYTHONPATH"] = str(FAILURETRACE) + os.pathsep + os.environ.get("PYTHONPATH", "")

import importlib
importlib.invalidate_caches()
import failuretrace
print("\nfailuretrace loaded from", failuretrace.__file__)

## 4 · Apply the GPU patches

In [ ]:
TOOLS = FAILURETRACE / "tools" / "lightning"

if PROFILE == "stock":
    print("Hopper detected — running autoresearch unmodified (FlashAttention-3 native).")
else:
    # FA3 -> torch SDPA. Sliding-window layers preserved via an additive causal-band mask.
    sh(sys.executable, TOOLS / "patch_train.py", AUTORESEARCH / "train.py")

if PROFILE == "t4":
    # depth 8->4, seq 2048->512, batch 128->8, bf16->fp16, EVAL_TOKENS 40->2 x 524288.
    sh(sys.executable, TOOLS / "patch_t4.py", AUTORESEARCH)

sh("git", "--no-pager", "diff", "--stat", cwd=AUTORESEARCH, check=False)

## 5 · Data prep (CPU-bound — costs no GPU credit)

`prepare.py` writes to `~/.cache/autoresearch`, which **does not persist** on Kaggle or
Colab. It is symlinked into the working directory below so it survives as notebook
output and can be re-attached as a Dataset on the next run.

The cached artifacts are the parquet shards + `tokenizer.pkl` + `token_bytes.pt`. None of
them depend on `MAX_SEQ_LEN` — that constant is only read at train time by
`make_dataloader` / `evaluate_bpb` (`prepare.py:353-354`). **So one cache is valid for
every profile**, T4 and A100 alike, despite the ordering note in the tools README.

In [ ]:
CACHE_HOME = Path.home() / ".cache" / "autoresearch"
PERSISTED  = ROOT / "autoresearch-cache"

# Re-use a previously prepared cache if one is attached (Kaggle Dataset / Colab Drive).
ATTACHED = next((p for p in [Path("/kaggle/input/autoresearch-cache"),
                             ROOT / "autoresearch-cache"]
                 if (p / "tokenizer" / "tokenizer.pkl").exists()), None)

if ATTACHED and ATTACHED != PERSISTED:
    print(f"re-using attached cache: {ATTACHED}")
    TARGET = ATTACHED
else:
    PERSISTED.mkdir(parents=True, exist_ok=True)
    TARGET = PERSISTED

CACHE_HOME.parent.mkdir(parents=True, exist_ok=True)
if CACHE_HOME.is_symlink():
    CACHE_HOME.unlink()
elif CACHE_HOME.exists():
    shutil.rmtree(CACHE_HOME)
CACHE_HOME.symlink_to(TARGET, target_is_directory=True)
print(f"~/.cache/autoresearch -> {TARGET}")

# Idempotent: prepare.py skips shards already on disk and a tokenizer already trained.
sh(sys.executable, "prepare.py", "--num-shards", SHARDS, cwd=AUTORESEARCH)

n_shards = len(list((TARGET / "data").glob("*.parquet")))
size_gb  = sum(f.stat().st_size for f in TARGET.rglob("*") if f.is_file()) / 1024**3
print(f"\ncache ready: {n_shards} shards, {size_gb:.2f} GiB at {TARGET}")

## 6 · Run the trials

Four real runs: a **baseline**, two **oversized-batch failures** on distinct seeds
(→ real CUDA OOM ×2 → `resource_pressure` → C1→C2 via the replication gate), and one
**reduced-batch fix** (the controlled counterfactual → C2→C3 + effect estimate).

`--launcher "python train.py"` overrides the default `uv run train.py`; `uv` is not
installed on Kaggle or Colab.

In [ ]:
DATA_DIR    = ROOT / f"ft_data_{PROFILE}"
REPORTS_DIR = ROOT / f"ft_reports_{PROFILE}"

cmd = [sys.executable, str(TOOLS / "run_trials.py"),
       "--repo", str(AUTORESEARCH),
       "--data-dir", str(DATA_DIR),
       "--reports-dir", str(REPORTS_DIR),
       "--launcher", "python train.py",
       "--profile", "t4" if PROFILE == "t4" else "a100"]

if CUSTOM_CONFIGS is not None:
    cfg_path = ROOT / "trial_configs.json"
    cfg_path.write_text(json.dumps(CUSTOM_CONFIGS, indent=2))
    cmd += ["--configs", str(cfg_path)]
    print("custom configs:", json.dumps([c["overrides"] for c in CUSTOM_CONFIGS]))

# Stream output: this takes ~30-45 min and silence is indistinguishable from a hang.
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print(f"\n[run_trials.py exit={proc.returncode}]")

## 7 · Inspect the real evidence

In [ ]:
print("=" * 72)
print("GOVERNANCE SUMMARY")
print("=" * 72)
summary = REPORTS_DIR / "summary.md"
print(summary.read_text() if summary.exists() else f"!! {summary} not written")

print("\n" + "=" * 72)
print("RAW run.log TAILS  (what the classifier actually parsed)")
print("=" * 72)
for log in sorted(REPORTS_DIR.glob("*.log")):
    tail = log.read_text(errors="replace").strip().splitlines()[-12:]
    print(f"\n--- {log.name} ---")
    print("\n".join(tail))

In [ ]:
# Independent cross-check straight from the store: what did the ladder actually reach?
from failuretrace import Repository, load_settings

settings = load_settings(overrides={"paths": {"data_dir": str(DATA_DIR),
                                              "reports_dir": str(REPORTS_DIR)},
                                    "ollama_enabled": False}, env={})
repo = Repository(settings)

for h in repo.list_hypotheses():
    level = repo.effective_causal_level(h.hypothesis_id)
    est   = repo.latest_effect_estimate(h.hypothesis_id)
    print(f"{h.category.value:20s} {level.value if level else '-':30s} "
          f"conf={h.hypothesis_confidence:.2f}"
          + (f"  effect={est.absolute_effect:+.4g} (n={est.n_counterfactuals})" if est else ""))

print(f"\ntrials stored: {len(repo.list_trials())}")

## 8 · Persist the cache so the next run skips prep

**Kaggle** — after this notebook finishes, its output contains `autoresearch-cache/`.
Save Version → then *Add Data → Your Datasets → New Dataset from this notebook's output*,
name it `autoresearch-cache`, and attach it to the next run. Cell 5 detects it at
`/kaggle/input/autoresearch-cache` and skips the 1 GB download entirely.

**Colab** — mount Drive and point `PERSISTED` at a Drive folder.

**Lightning** — Studio storage already persists; run cells 1–5 on a **free CPU Studio**,
then switch the same Studio to A100 and run cells 6–7. Prep burns no GPU credit that way.

---

## What this does and does not establish

**Does:** the `run.log` parser handles a real log; a real `torch.OutOfMemoryError`
traceback classifies as `resource_pressure` deterministically; the replication gate
promotes C1→C2 across distinct (seed, commit) units on real evidence; a real controlled
counterfactual reaches C3 with a measured effect.

**Does not:** show that FailureTrace saves compute. The configs deliberately engineer
OOMs because that is the one deterministic, *plannable* route to C3. Presenting this as
value validation would be exactly the C0/C1-as-causal error the project exists to prevent.
That claim needs the replay study (offline, CPU-only) and then a placebo-controlled A/B.

## Troubleshooting

| Symptom | Cause / fix |
|---|---|
| `expected pinned autoresearch text not found` | Upstream moved past `228791f`. Check out the pin, or update `_OLD`/`_TRAIN_REPLACEMENTS`. |
| Baseline produces no `val_bpb`, run aborts | Baseline OOMed. Lower `base` in cell 2 — this abort is deliberate: no baseline means no valid comparison, so no failure evidence is written. |
| Very long pause before step 0 | `torch.compile(dynamic=False, fullgraph=True)` (`train.py:508`) — 1–3 min, and it sits *outside* the 5-min budget. |
| T4 crashes inside the Muon optimizer | fp16 + `@torch.compile(fullgraph=True)` Newton-Schulz (`train.py:305-324`). Turing has no native bf16. This classifies as a genuine `runtime_failure` trial, not a FailureTrace bug. |
| `No CUDA GPU visible` | Accelerator not enabled. On Kaggle also confirm **Internet: On** or the shard download fails. |
| Kaggle re-downloads data every session | `~/.cache` is not persisted — attach the cache Dataset (above). |